# 01 - Predicción de mora crediticia con Logistic Regression

## Caso fintech
Este notebook simula un problema de riesgo crediticio: predecir si un cliente caerá en mora.

## Objetivo
Entrenar un modelo de clasificación binaria usando Logistic Regression.

## Dataset
Se genera un dataset sintético de 2,000 clientes con variables financieras y comportamentales.

## Técnicas
- Análisis exploratorio
- Limpieza y preparación de datos
- One-hot encoding
- Escalamiento
- Logistic Regression
- Matriz de confusión
- ROC-AUC
- Precision, Recall y F1-score


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay
)

np.random.seed(42)


In [ ]:
# Generación de dataset sintético
n = 2000

age = np.random.randint(21, 70, n)
monthly_income = np.random.normal(4200, 1800, n).clip(1200, 16000)
credit_score = np.random.normal(640, 90, n).clip(300, 850)
debt_to_income = np.random.beta(2, 5, n)
num_loans = np.random.poisson(1.8, n)
late_payments_12m = np.random.poisson(0.8, n)
credit_card_utilization = np.random.beta(2.5, 3.5, n)
employment_years = np.random.exponential(4, n).clip(0, 30)
customer_segment = np.random.choice(["mass", "premium", "young", "microbusiness"], n, p=[0.45, 0.2, 0.25, 0.1])

# Probabilidad sintética de mora
risk_score = (
    -3.2
    + 2.8 * debt_to_income
    + 2.2 * credit_card_utilization
    + 0.45 * late_payments_12m
    + 0.12 * num_loans
    - 0.004 * (credit_score - 600)
    - 0.00008 * (monthly_income - 4000)
    - 0.03 * employment_years
    + np.where(customer_segment == "microbusiness", 0.35, 0)
    + np.where(customer_segment == "premium", -0.35, 0)
)

prob_default = 1 / (1 + np.exp(-risk_score))
default = np.random.binomial(1, prob_default)

df = pd.DataFrame({
    "age": age,
    "monthly_income": monthly_income.round(2),
    "credit_score": credit_score.round(0),
    "debt_to_income": debt_to_income.round(3),
    "num_loans": num_loans,
    "late_payments_12m": late_payments_12m,
    "credit_card_utilization": credit_card_utilization.round(3),
    "employment_years": employment_years.round(1),
    "customer_segment": customer_segment,
    "default": default
})

df.head()


In [ ]:
df.shape, df["default"].value_counts(normalize=True).round(3)

In [ ]:
df.describe(include="all")

In [ ]:
# Visualización simple de variables relevantes
df[["monthly_income", "credit_score", "debt_to_income", "credit_card_utilization"]].hist(figsize=(10, 8))
plt.tight_layout()
plt.show()


In [ ]:
X = df.drop(columns=["default"])
y = df["default"]

numeric_features = [
    "age", "monthly_income", "credit_score", "debt_to_income", "num_loans",
    "late_payments_12m", "credit_card_utilization", "employment_years"
]
categorical_features = ["customer_segment"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))


In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("Curva ROC - Predicción de mora")
plt.show()


In [ ]:
# Coeficientes del modelo para interpretación
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
}).sort_values("coefficient", ascending=False)

coef_df
